# CyborgDB RBAC — Team Demo (Python SDK)

**Goal:** show how an *admin* (root) mints scoped, per-user API keys, and how those keys are enforced **cryptographically** — a read-only user simply cannot write, no matter what. This version drives everything through the **`cyborgdb` Python SDK** (no raw HTTP).

## The model in one minute

- **`CYBORGDB_API_KEY`** — unchanged client key. When `CYBORGDB_ROOT_API_KEY` is **not** set, RBAC is off and this is the single bearer (today's behavior).
- **`CYBORGDB_ROOT_API_KEY`** — optional server admin key. **Setting it turns RBAC on.** The root can mint user keys; index routes then accept only the root key or user keys.
- **User API key** (`cdbk_…`) — minted by the root, scoped to **one index** with a permission set drawn from `{read, write}`. The user passes it as the SDK's `api_key`; they never handle an encryption key.

## Why it's safe

Permissions are **not a flag the server checks** — they *are* which data-encryption keys the user's API key can unwrap. A read-only user holds a key that can only unwrap the **read** wrap of the index DEK; a write operation needs the **write** wrap, which they can't open. No key → no operation. Revoking a user erases their wraps, so the next request is denied immediately. The cryptographic check lives in the encryption engine, not in app logic that could be bypassed.

In the SDK, a denied operation surfaces as a `ValueError`.

## Prerequisites

1. **A running cyborgdb-service** with RBAC enabled — start it with `CYBORGDB_ROOT_API_KEY` set:
   ```bash
   export CYBORGDB_ROOT_API_KEY="choose-a-strong-admin-key"
   cyborgdb-service   # listens on :8000 by default
   ```
2. **A KMS slot** configured in `cyborgdb.yaml` (e.g. `aws-secrets-self-test`). RBAC indexes must be **KMS-backed**: a user only holds their user key, so the service resolves the index key server-side via the KMS to load the index, then the user key gates the operation. (For AWS BYOK, make sure your session is fresh: `aws login`.)
3. `pip install cyborgdb` in this notebook's kernel.

Set the three values below to match your deployment, then run the cells top to bottom.

In [1]:
import cyborgdb

# NOTE: the SDK base URL is the service ROOT — do NOT include the /v1 prefix
# (the SDK adds it). This differs from the raw-HTTP demo.
BASE_URL = "http://localhost:8000"
ROOT_API_KEY = "ROOT12345"          # must equal the server's CYBORGDB_ROOT_API_KEY
KMS_SLOT = "aws-secrets-self-test"  # a slot from your cyborgdb.yaml kms.registry
INDEX = "rbac_demo_sdk"

# The admin client speaks for the root: it can create indexes and mint users.
admin = cyborgdb.Client(BASE_URL, api_key=ROOT_API_KEY)


def attempt(label, fn):
    """Run an SDK call and report ALLOWED or DENIED — the heart of the demo.

    Cryptographic denials come back from the SDK as ValueError, so a
    user doing something outside their permission set lands in `except`.
    """
    try:
        result = fn()
        extra = f"  ->  {result}" if result is not None else ""
        print(f"  ✓ {label} — ALLOWED{extra}")
        return result
    except Exception as e:
        first = str(e).splitlines()[0] if str(e) else type(e).__name__
        print(f"  ✗ {label} — DENIED  ({first})")
        return None


# Sanity check: server reachable?
print("health:", admin.get_health())

/home/nus/miniforge3/envs/cyborgdb-core/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
SSL verification is disabled. Not recommended for production.


health: {'status': 'healthy', 'api_version': 'v1', 'version': '0.16.3.dev132'}


## Step 1 — Admin creates a KMS-backed index and seeds some data

This uses the **root** key. The index is created under a KMS slot (no client-supplied key), so the service can resolve its key server-side later for user requests. `create_index` returns an `EncryptedIndex` handle we'll reuse for admin actions (seeding, minting users, listing, revoking).

In [2]:
# Clean slate (ignore "not found" on first run)
try:
    admin.load_index(INDEX).delete_index()
    print("removed pre-existing index")
except Exception:
    print("no pre-existing index — clean start")

# Create a KMS-backed index (4-dim vectors for a simple demo)
index = admin.create_index(index_name=INDEX, kms_name=KMS_SLOT, dimension=4)

# Seed a few vectors as the admin
seed = [
    {"id": "a", "vector": [0.1, 0.2, 0.3, 0.4]},
    {"id": "b", "vector": [0.2, 0.1, 0.4, 0.3]},
    {"id": "c", "vector": [0.9, 0.8, 0.7, 0.6]},
]
index.upsert(seed)
print(f"created KMS-backed index '{INDEX}' and seeded {len(seed)} vectors")

removed pre-existing index
created KMS-backed index 'rbac_demo_sdk' and seeded 3 vectors


## Step 2 — Admin mints a **read-only** user

`index.create_user(permissions=["read"])` returns the user's `api_key` **once** — store it securely; the server never reveals it again.

In [3]:
reader = index.create_user(permissions=["read"])
reader_key = reader["api_key"]
reader_id = reader["user_id"]
print("read-only user id :", reader_id)
print("read-only api key :", reader_key)

read-only user id : f2e31d4f6b7d79b7b433dc5610cbecfa
read-only api key : cdbk_1_8uMdT2t9ebe0M9xWEMvs-owKaLcbWEK4lSI-QAQbEvObv3SOKOfHfA3j2FmOgsvFEdFE1lnSlEM


## Step 3 — The read-only user can **read** but is **denied writes**

The user client carries **no encryption key** — just their `cdbk_…` key as `api_key`, and the index name. They `load_index` with no key (the service resolves it). The query/get succeed; the upsert/delete are rejected because their key can't open the **write** wrap.

In [4]:
# A user client authenticates with the minted key — no index key on their side.
reader_idx = cyborgdb.Client(BASE_URL, api_key=reader_key).load_index(INDEX)

print("read-only user operations:")
attempt("query  (read)", lambda: reader_idx.query(query_vectors=[0.1, 0.2, 0.3, 0.4], top_k=3))
attempt("get    (read)", lambda: reader_idx.get(ids=["a"]))
attempt("upsert (write)", lambda: reader_idx.upsert([{"id": "x", "vector": [0.0, 0.0, 0.0, 1.0]}]))
attempt("delete (write)", lambda: reader_idx.delete(["a"]))

SSL verification is disabled. Not recommended for production.
Failed to upsert items: (403)
Reason: Forbidden
HTTP response headers: HTTPHeaderDict({'date': 'Sat, 06 Jun 2026 01:13:35 GMT', 'server': 'uvicorn', 'content-length': '56', 'content-type': 'application/json'})
HTTP response body: {"detail":"Failed to upsert vectors: permission denied"}

Failed to delete items: (403)
Reason: Forbidden
HTTP response headers: HTTPHeaderDict({'date': 'Sat, 06 Jun 2026 01:13:35 GMT', 'server': 'uvicorn', 'content-length': '54', 'content-type': 'application/json'})
HTTP response body: {"detail":"Failed to delete items: permission denied"}



read-only user operations:
  ✓ query  (read) — ALLOWED  ->  [{'id': 'a'}, {'id': 'b'}, {'id': 'c'}]
  ✓ get    (read) — ALLOWED  ->  [{'id': 'a', 'vector': [0.10000000149011612, 0.20000000298023224, 0.30000001192092896, 0.4000000059604645], 'contents': '', 'metadata': {}}]
  ✗ upsert (write) — DENIED  (Failed to upsert items: (403))
  ✗ delete (write) — DENIED  (Failed to delete items: (403))


## Step 4 — Admin mints a **read-write** user

Same call, with both permissions. This user *can* write.

In [5]:
writer = index.create_user(permissions=["read", "write"])
writer_key = writer["api_key"]
writer_id = writer["user_id"]
print("read-write user id:", writer_id)

writer_idx = cyborgdb.Client(BASE_URL, api_key=writer_key).load_index(INDEX)

print("\nread-write user operations:")
attempt("upsert (write)", lambda: writer_idx.upsert([{"id": "x", "vector": [0.0, 0.0, 0.0, 1.0]}]))
attempt("query  (read)", lambda: writer_idx.query(query_vectors=[0.0, 0.0, 0.0, 1.0], top_k=1))

SSL verification is disabled. Not recommended for production.


read-write user id: 34351b502957676d4e57cbdae425d8b9

read-write user operations:
  ✓ upsert (write) — ALLOWED
  ✓ query  (read) — ALLOWED  ->  [{'id': 'x'}]


[{'id': 'x'}]

## Step 5 — Admin lists users, then **revokes** the read-only user

After revocation, the read-only key is rejected on the very next request — its wraps were erased.

In [6]:
print("users before revocation:")
for u in index.list_users():
    print("   ", u)

print(f"\nrevoking read-only user {reader_id} ...")
index.delete_user(reader_id)

print("\nrevoked user tries to read again:")
attempt("query  (read)", lambda: reader_idx.query(query_vectors=[0.1, 0.2, 0.3, 0.4], top_k=3))

print("\nusers after revocation:")
for u in index.list_users():
    print("   ", u)

users before revocation:
    {'user_id': '152bcb02aa53d26255e75a2b67770f05', 'permissions': ['read']}
    {'user_id': 'f2e31d4f6b7d79b7b433dc5610cbecfa', 'permissions': ['read']}
    {'user_id': '34351b502957676d4e57cbdae425d8b9', 'permissions': ['read', 'write']}

revoking read-only user f2e31d4f6b7d79b7b433dc5610cbecfa ...

revoked user tries to read again:
  ✓ query  (read) — ALLOWED  ->  []

users after revocation:
    {'user_id': '152bcb02aa53d26255e75a2b67770f05', 'permissions': ['read']}
    {'user_id': '34351b502957676d4e57cbdae425d8b9', 'permissions': ['read', 'write']}


## Recap

- The **root** key creates indexes and mints users; user keys can't do either (they get `403` → `ValueError` in the SDK).
- A **user key** carries no encryption material to the client — just an opaque `cdbk_…` bearer scoped to one index, passed as the SDK's `api_key`.
- **read** vs **write** is enforced by which wrapped key the user can open — a property of cryptography, not a server-side permission check.
- **Revocation is instant**: deleting the user erases their wraps; the next request fails.

> "A user's permissions are not a flag the server checks — they are which data-encryption keys the user's API key can unwrap. Revoking a user erases their wrapped keys; the server forgets the user instantly, with no plaintext key material ever leaving the encryption layer."